In [1]:
import os, shutil
src = None
for root, dirs, files in os.walk("/kaggle/input"):
    dirs[:] = [d for d in dirs if d != "competitions"]
    if "10_train_student_d32.py" in files:
        src = os.path.join(root, "10_train_student_d32.py"); break
assert src, "10_train_student_d32.py tidak ditemukan; tambahkan input rsna-knee-14-student-d32"
shutil.copy(src, "/kaggle/working/10_train_student_d32.py")
print("Script disalin dari:", src)

Script disalin dari: /kaggle/input/notebooks/kellenvalerie/rsna-knee-14-student-d32/10_train_student_d32.py


In [2]:
%%bash
cd /kaggle/working
COMMON="--alpha 0.5 --oof oof_effb0_kf2.csv --meta _meta.csv --backbone vit_small_patch14_dinov2.lvd142m \
        --lr_backbone 1e-4 --llrd 0.8 --batch 4 --epochs 10 --nfolds 5 --tag dinov2s_kf"

( for f in 0 1 2; do
    CUDA_VISIBLE_DEVICES=0 python -u 10_train_student_d32.py $COMMON --fold $f > log_dinov2s_kf_f$f.txt 2>&1
  done ) &
( for f in 3 4; do
    CUDA_VISIBLE_DEVICES=1 python -u 10_train_student_d32.py $COMMON --fold $f > log_dinov2s_kf_f$f.txt 2>&1
  done ) &
wait
for f in 0 1 2 3 4; do echo "===== fold $f ====="; tail -n 3 log_dinov2s_kf_f$f.txt; done
grep -H "Train \|Label  :\|Label diperbaiki\|Error\|error" log_dinov2s_kf_f0.txt log_dinov2s_kf_f3.txt

===== fold 0 =====
Contusion          AUC=0.866
Fracture           AUC=0.903
Mean AUC gold: 0.8593
===== fold 1 =====
Contusion          AUC=0.845
Fracture           AUC=0.846
Mean AUC gold: 0.8527
===== fold 2 =====
Contusion          AUC=0.865
Fracture           AUC=0.879
Mean AUC gold: 0.8630
===== fold 3 =====
Contusion          AUC=0.862
Fracture           AUC=0.900
Mean AUC gold: 0.8544
===== fold 4 =====
Contusion          AUC=0.870
Fracture           AUC=0.883
Mean AUC gold: 0.8533
log_dinov2s_kf_f0.txt:Label  : /kaggle/input/notebooks/kellenvalerie/rsna-knee-01d-label-ensemble/report_labels.csv
log_dinov2s_kf_f0.txt:Label diperbaiki: alpha=0.5 (guru: oof_effb0_kf2.csv)
log_dinov2s_kf_f0.txt:Train 3479 | Val 870 | Gold holdout 58
log_dinov2s_kf_f3.txt:Label  : /kaggle/input/notebooks/kellenvalerie/rsna-knee-01d-label-ensemble/report_labels.csv
log_dinov2s_kf_f3.txt:Label diperbaiki: alpha=0.5 (guru: oof_effb0_kf2.csv)
log_dinov2s_kf_f3.txt:Train 3479 | Val 870 | Gold holdout 58

In [3]:
import os, glob, numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score

COMP = "/kaggle/input/competitions/rsna-knee-abnormality-detection"
LABELS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA",
          "Lateral OA", "PF OA", "Effusion", "Synovitis", "Baker's",
          "Contusion", "Fracture"]

def find(name):
    for root, dirs, files in os.walk("/kaggle/input"):
        dirs[:] = [d for d in dirs if d != "competitions"]
        if name in files:
            return os.path.join(root, name)
    raise FileNotFoundError(name)

def find_all(pattern_dir, prefix):
    out = []
    for root, dirs, files in os.walk(pattern_dir):
        dirs[:] = [d for d in dirs if d != "competitions"]
        out += [os.path.join(root, f) for f in files if f.startswith(prefix) and f.endswith(".csv")]
    return sorted(out)

def mean_auc(y, p):
    aucs = []
    for k in range(len(LABELS)):
        ok = ~np.isnan(y[:, k]) & (y[:, k] != 0.5)
        yk = (y[ok, k] > 0.5).astype(float)
        aucs.append(roc_auc_score(yk, p[ok, k]) if len(np.unique(yk)) == 2 else np.nan)
    return float(np.nanmean(aucs))

# --- OOF guru baru (DINOv2) dan guru lama (EffB0) ---
oof_d = pd.concat([pd.read_csv(f) for f in sorted(glob.glob("/kaggle/working/val_preds_dinov2s_kf_f*.csv"))])
oof_e = pd.read_csv(find("oof_effb0_kf2.csv"))
print(f"OOF DINOv2: {len(oof_d)} studi (unik {oof_d['StudyInstanceUID'].nunique()}) | OOF EffB0: {len(oof_e)}")
m = oof_d.merge(oof_e, on="StudyInstanceUID", suffixes=("_d", "_e"))
teacher2 = m[["StudyInstanceUID"]].copy()
for l in LABELS:
    teacher2[l] = (m[l + "_d"] + m[l + "_e"]) / 2
teacher2.to_csv("/kaggle/working/oof_teacher2.csv", index=False)
oof_d.to_csv("/kaggle/working/oof_dinov2s_kf.csv", index=False)
print("Tersimpan: oof_teacher2.csv dan oof_dinov2s_kf.csv |", len(teacher2), "studi")

# --- prediksi gold: rata-rata 5 fold untuk setiap guru ---
tr = pd.read_csv(f"{COMP}/train.csv")
gold = tr[tr[LABELS].notna().all(axis=1)][["StudyInstanceUID"] + LABELS].reset_index(drop=True)
Yg = gold[LABELS].values.astype(float)

def gold_mean(files):
    return np.mean([gold[["StudyInstanceUID"]].merge(pd.read_csv(f), on="StudyInstanceUID")[LABELS].values
                    for f in files], axis=0)

g_d = gold_mean(sorted(glob.glob("/kaggle/working/gold_preds_dinov2s_kf_f*.csv")))
g_e = gold_mean(find_all("/kaggle/input", "gold_preds_effb0_kf2_f"))
g_t2 = (g_d + g_e) / 2
llm = pd.read_csv(find("report_labels.csv")).set_index("StudyInstanceUID").loc[gold["StudyInstanceUID"], LABELS].values.astype(float)
llm = np.nan_to_num(llm, nan=0.5)

print(f"\nGuru lama EffB0 5-fold     gold {mean_auc(Yg, g_e):.4f}")
print(f"Guru baru DINOv2 5-fold    gold {mean_auc(Yg, g_d):.4f}")
print(f"Guru gabungan (teacher2)   gold {mean_auc(Yg, g_t2):.4f}")

print("\nKualitas LABEL LATIH terhadap gold  (label = w x LLM + (1-w) x guru)")
for w in [0.3, 0.4, 0.5]:
    print(f"  w={w:.1f} | label sekarang (guru EffB0): {mean_auc(Yg, w*llm + (1-w)*g_e):.4f}"
          f" | label baru (teacher2): {mean_auc(Yg, w*llm + (1-w)*g_t2):.4f}")

OOF DINOv2: 4349 studi (unik 4349) | OOF EffB0: 4349
Tersimpan: oof_teacher2.csv dan oof_dinov2s_kf.csv | 4349 studi

Guru lama EffB0 5-fold     gold 0.8512
Guru baru DINOv2 5-fold    gold 0.8658
Guru gabungan (teacher2)   gold 0.8648

Kualitas LABEL LATIH terhadap gold  (label = w x LLM + (1-w) x guru)
  w=0.3 | label sekarang (guru EffB0): 0.8975 | label baru (teacher2): 0.9039
  w=0.4 | label sekarang (guru EffB0): 0.9021 | label baru (teacher2): 0.9060
  w=0.5 | label sekarang (guru EffB0): 0.9020 | label baru (teacher2): 0.9056
